In [ ]:
import os
# PyTorch >= 2.6 loads files with weights_only=True by default, which rejects the
# averaging_snapshots.pkl that SuperGradients writes and re-reads during training
# ("average_best_models": True) and makes trainer.train() crash after the second epoch.
# Restore the old default; this run only loads its own checkpoints and the official COCO weights.
os.environ.setdefault("TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD", "1")

import sys
import torch
from pathlib import Path
 
# Check if GPU is available 
DEVICE = 'cuda' if torch.cuda.is_available() else "cpu"


def find_project_dir() -> Path:
    """Locate the yolonas-training/ folder (the one holding this notebook), so that
    paths do not depend on the directory the kernel happens to be started in."""
    candidates = []
    # VS Code exposes the notebook's path as __vsc_ipynb_file__; Jupyter as JPY_SESSION_NAME.
    notebook_file = globals().get("__vsc_ipynb_file__") or os.environ.get("JPY_SESSION_NAME")
    if notebook_file:
        candidates.append(Path(notebook_file).resolve().parent)
    cwd = Path.cwd().resolve()
    for folder in (cwd, *cwd.parents):
        candidates += [folder, folder / "yolonas-training"]
    for candidate in candidates:
        if (candidate / "train.ipynb").is_file() and (candidate / "training-data").is_dir():
            return candidate
    raise FileNotFoundError(
        f"Could not find the yolonas-training/ folder (train.ipynb + training-data/) from {cwd}. "
        "Start the kernel from inside the defect-detection repo."
    )


# project directory (yolonas-training/); every path below is built from it
HOME = str(find_project_dir())
if HOME not in sys.path:
    sys.path.insert(0, HOME)  # so `from copy_paste_aug import ...` works from any working directory

EXPERIMENT_NAME="building_defect_yolo_nas_m"
# models/ is the shared top-level directory (next to yolonas-training/),
# also read by batch_inference. SuperGradients writes checkpoints to
# {CHECKPOINT_DIR}/{EXPERIMENT_NAME}/RUN_*/...
CHECKPOINT_DIR = str(Path(HOME).parent / "models")

print(f"Training data : {HOME}/training-data")
print(f"Checkpoints   : {CHECKPOINT_DIR}/{EXPERIMENT_NAME}/RUN_*/")

In [ ]:
from super_gradients.training import Trainer
 
""" 
Trainer --> Initialize the Trainer for the YOLO-NAS model
Trainer is responsible for managing the training process, including data loading, model training, and evaluation.
trainer is an instance of the Trainer class from the super_gradients.training module.
"""
trainer = Trainer(experiment_name=EXPERIMENT_NAME, ckpt_root_dir=CHECKPOINT_DIR)
  
IMG_SIZE = [640, 640]  # Input image size for the model

# Dataset , Label information
dataset_params = {
    'data_dir': f"{HOME}/training-data",
    'train_images_dir':'train/images',
    'train_labels_dir':'train/labels',
    'val_images_dir':'val/images',
    'val_labels_dir':'val/labels',
    'classes': [line.strip() for line in open(f"{HOME}/training-data/classes.txt") if line.strip()], # labels here
    'input_dim': IMG_SIZE
}

In [ ]:
from super_gradients.training.dataloaders.dataloaders import (
    coco_detection_yolo_format_train, coco_detection_yolo_format_val)
 
# you can increase this number depending on your GPU, more batch size means faster training
BATCH_SIZE = 8
NUM_WORKERS = min(8, max(2, os.cpu_count() or 2))

train_transforms = [
    {
        "DetectionMosaic": {
            "input_dim": IMG_SIZE,
            "prob": 0.7
        }
    },
    {
        "DetectionRandomAffine": {
            "degrees": 10.0,
            "translate": 0.1,
            "scales": [0.5, 1.5],
            "shear": 2.0,
            "target_size": IMG_SIZE,
            "filter_box_candidates": True,
            "wh_thr": 2,
            "area_thr": 0.1,
            "ar_thr": 20
        }
    },
    {
        "DetectionMixup": {
            "input_dim": IMG_SIZE,
            "mixup_scale": [0.5, 1.5],
            "prob": 0.2,
            "flip_prob": 0.5
        }
    },
    {
        "DetectionHSV": {
            "prob": 0.5,
            "hgain": 5,
            "sgain": 30,
            "vgain": 30
        }
    },
    {
        "DetectionHorizontalFlip": {
            "prob": 0.5
        }
    },
    {
        "DetectionPaddedRescale": {
            "input_dim": IMG_SIZE
        }
    },
    {
        "DetectionTargetsFormatTransform": {
            "input_dim": IMG_SIZE,
            "output_format": "LABEL_CXCYWH"
        }
    }
]

val_transforms = [
    {
        "DetectionPaddedRescale": {
            "input_dim": IMG_SIZE
        }
    },
    {
        "DetectionTargetsFormatTransform": {
            "input_dim": IMG_SIZE,
            "output_format": "LABEL_CXCYWH"
        }
    }
]
 
train_data = coco_detection_yolo_format_train(
    dataset_params={
        'data_dir': dataset_params['data_dir'],
        'images_dir': dataset_params['train_images_dir'],
        'labels_dir': dataset_params['train_labels_dir'],
        'classes': dataset_params['classes'],
        'transforms': train_transforms
    },
    dataloader_params={
        'batch_size': BATCH_SIZE,
        'num_workers': NUM_WORKERS
    }
)
 
val_data = coco_detection_yolo_format_val(
    dataset_params={
        'data_dir': dataset_params['data_dir'],
        'images_dir': dataset_params['val_images_dir'],
        'labels_dir': dataset_params['val_labels_dir'],
        'classes': dataset_params['classes'],
        'transforms': val_transforms
    },
    dataloader_params={
        'batch_size': BATCH_SIZE,
        'num_workers': NUM_WORKERS
    }
)


In [ ]:
from copy_paste_aug import ClassAwareCopyPaste

USE_COPY_PASTE = False  # disabled by default; set True to enable ClassAwareCopyPaste (see copy_paste_aug.py)

if USE_COPY_PASTE:
    _copy_paste = ClassAwareCopyPaste(
        images_dir=f"{dataset_params['data_dir']}/{dataset_params['train_images_dir']}",
        labels_dir=f"{dataset_params['data_dir']}/{dataset_params['train_labels_dir']}",
        rare_classes=[0, 2, 3],  # corrosion, spall, plant_growth
        paste_prob=0.5,
        max_paste=3,
        scale_range=(0.05, 0.25),
        max_overlap=0.1,
    )
    train_data = coco_detection_yolo_format_train(
        dataset_params={
            'data_dir': dataset_params['data_dir'],
            'images_dir': dataset_params['train_images_dir'],
            'labels_dir': dataset_params['train_labels_dir'],
            'classes': dataset_params['classes'],
            'transforms': [_copy_paste] + train_transforms
        },
        dataloader_params={
            'batch_size': BATCH_SIZE,
            'num_workers': NUM_WORKERS,
        }
    )
    print("Copy-paste augmentation enabled.")

    # --- Preview ---
    import cv2
    import numpy as np
    import matplotlib.pyplot as plt
    import random
    from pathlib import Path
    from super_gradients.training.samples import DetectionSample

    _img_dir = Path(f"{dataset_params['data_dir']}/{dataset_params['train_images_dir']}")
    _lbl_dir = Path(f"{dataset_params['data_dir']}/{dataset_params['train_labels_dir']}")
    _class_names = dataset_params['classes']
    _colors = [(220, 50, 50), (50, 200, 50), (50, 50, 220), (200, 180, 0), (0, 180, 200)]

    _img_files = [p for p in _img_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}]
    _img_path = random.choice(_img_files)
    _lbl_path = _lbl_dir / (_img_path.stem + ".txt")

    _img = cv2.imread(str(_img_path))
    _H, _W = _img.shape[:2]

    _bboxes, _labels = [], []
    if _lbl_path.exists():
        for _line in _lbl_path.read_text().strip().splitlines():
            if not _line.strip():
                continue
            _p = _line.split()
            _cls = int(_p[0])
            _cx, _cy, _w, _h = float(_p[1]), float(_p[2]), float(_p[3]), float(_p[4])
            _bboxes.append([(_cx - _w/2)*_W, (_cy - _h/2)*_H, (_cx + _w/2)*_W, (_cy + _h/2)*_H])
            _labels.append(_cls)

    _orig_sample = DetectionSample(
        image=_img,
        bboxes_xyxy=np.array(_bboxes, dtype=np.float32).reshape(-1, 4) if _bboxes else np.zeros((0, 4), dtype=np.float32),
        labels=np.array(_labels, dtype=np.int32),
    )

    # Retry until at least one instance is pasted (paste_prob may not fire every call)
    _aug_sample = _orig_sample
    for _ in range(20):
        _aug_sample = _copy_paste.apply_to_sample(_orig_sample)
        if len(_aug_sample.labels) > len(_orig_sample.labels):
            break

    def _draw_boxes(img, bboxes, labels, class_names, colors):
        out = img.copy()
        for bbox, label in zip(bboxes, labels):
            x1, y1, x2, y2 = map(int, bbox)
            color = colors[label % len(colors)]
            cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
            cv2.putText(out, class_names[label], (x1, max(12, y1 - 4)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2, cv2.LINE_AA)
        return out

    _vis_orig = _draw_boxes(_img, _orig_sample.bboxes_xyxy, _orig_sample.labels, _class_names, _colors)
    _vis_aug  = _draw_boxes(_aug_sample.image, _aug_sample.bboxes_xyxy, _aug_sample.labels, _class_names, _colors)

    _n_pasted = len(_aug_sample.labels) - len(_orig_sample.labels)
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    axes[0].imshow(cv2.cvtColor(_vis_orig, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f"Original  ({len(_orig_sample.labels)} annotations)", fontsize=13)
    axes[0].axis('off')
    axes[1].imshow(cv2.cvtColor(_vis_aug, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f"After copy-paste  (+{_n_pasted} pasted)", fontsize=13)
    axes[1].axis('off')
    plt.suptitle(_img_path.name, fontsize=11, y=1.01)
    plt.tight_layout()
    plt.show()

else:
    print("Copy-paste augmentation disabled.")

In [ ]:
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler
from tqdm.auto import tqdm

USE_WEIGHTED_SAMPLING = False

if USE_WEIGHTED_SAMPLING:
    dataset = train_data.dataset
    num_classes = len(dataset_params["classes"])

    def extract_labels(sample):
        # common shapes in SG detection datasets
        if isinstance(sample, (tuple, list)) and len(sample) >= 2:
            target = sample[1]
        elif isinstance(sample, dict):
            target = sample.get("targets") or sample.get("target") or sample.get("labels")
        else:
            target = sample

        # target can be Nx5 (class, cx, cy, w, h) or dict with "labels"
        if isinstance(target, dict) and "labels" in target:
            labels = target["labels"]
        else:
            labels = target[:, 0] if hasattr(target, "shape") and target.ndim == 2 else target

        if hasattr(labels, "detach"):
            labels = labels.detach().cpu()
        return torch.as_tensor(labels, dtype=torch.long)

    print("Counting class occurrences per box...")
    class_counts = torch.zeros(num_classes)
    for i in tqdm(range(len(dataset)), desc="Scan dataset (class counts)"):
        labels = extract_labels(dataset[i])
        if labels.numel() > 0:
            class_counts.scatter_add_(0, labels, torch.ones_like(labels, dtype=class_counts.dtype))

    print("Class counts:", class_counts.tolist())

    print("Computing inverse-frequency class weights...")
    class_weights = class_counts.sum() / torch.clamp(class_counts, min=1)
    print("Class weights:", class_weights.tolist())

    print("Assigning per-image sampling weights...")
    sample_weights = []
    for i in tqdm(range(len(dataset)), desc="Compute sample weights"):
        labels = extract_labels(dataset[i])
        w = 0.0 if labels.numel() == 0 else class_weights[labels].max().item()
        sample_weights.append(w)

    sample_weights = torch.tensor(sample_weights, dtype=torch.float)
    print("Sample weights summary:",
          f"min={sample_weights.min().item():.4f},",
          f"max={sample_weights.max().item():.4f},",
          f"mean={sample_weights.mean().item():.4f}")

    print("Building WeightedRandomSampler...")
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

    print("Rebuilding train DataLoader with sampler...")
    train_data = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        sampler=sampler,
        num_workers=NUM_WORKERS,
        collate_fn=train_data.collate_fn,
        pin_memory=True,
    )

    print("Weighted sampling enabled.")
else:
    dataset = train_data.dataset
    train_data = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        collate_fn=train_data.collate_fn,
        pin_memory=True,
    )
    print("Weighted sampling disabled — using default DataLoader.")


In [ ]:
from super_gradients.training import models

MODEL_ARCH = "yolo_nas_m"

# Choose initialization: "coco" for COCO pretrained or "checkpoint" to fine-tune from a checkpoint
INIT_SOURCE = "coco"  # "coco" or "checkpoint"
CHECKPOINT_PATH = ""  # set to a .pth path when INIT_SOURCE="checkpoint", e.g. f"{CHECKPOINT_DIR}/building_defect_yolo_nas_m/RUN_<id>/ckpt_best.pth"

if INIT_SOURCE == "coco":
    model = models.get(
        MODEL_ARCH,
        num_classes=len(dataset_params['classes']),
        pretrained_weights="coco"
    )
elif INIT_SOURCE == "checkpoint":
    if not CHECKPOINT_PATH:
        raise ValueError("CHECKPOINT_PATH must be set when INIT_SOURCE='checkpoint'")
    model = models.get(
        MODEL_ARCH,
        num_classes=len(dataset_params['classes']),
        checkpoint_path=CHECKPOINT_PATH
    )
else:
    raise ValueError(f"Unknown INIT_SOURCE: {INIT_SOURCE}")


In [ ]:
from super_gradients.training.losses import PPYoloELoss
from super_gradients.training.metrics import DetectionMetrics_050
from super_gradients.training.models.detection_models.pp_yolo_e import PPYoloEPostPredictionCallback
 
# Epoch Number
MAX_EPOCHS = 100

# Resume an interrupted run: set RESUME_PATH to that run's ckpt_latest.pth and RESUME_TRAINING to True
RESUME_PATH = ""  # e.g. f"{CHECKPOINT_DIR}/building_defect_yolo_nas_m/RUN_<id>/ckpt_latest.pth"
RESUME_TRAINING = False  # Set to True to resume training from the checkpoint
if RESUME_TRAINING and not RESUME_PATH:
    raise ValueError("RESUME_PATH must be set when RESUME_TRAINING=True")
 
train_params = {
    'silent_mode': False,
    "average_best_models":True,
    "warmup_mode": "linear_epoch_step",
    "warmup_initial_lr": 1e-6,
    "lr_warmup_epochs": 3,
    "initial_lr": 5e-4,
    "lr_mode": "cosine",
    "cosine_final_lr_ratio": 0.1,
    "batch_accumulate": 4,
 
    "optimizer": "AdamW",
    "optimizer_params": {"weight_decay": 0.0001},
    "zero_weight_decay_on_bias_and_bn": True,
    "ema": True,
    "ema_params": {"decay": 0.9999, "decay_type": "threshold"},
    "max_epochs": MAX_EPOCHS,
    "resume": RESUME_TRAINING,
    "resume_path": RESUME_PATH if RESUME_TRAINING else None,
    "resume_strict_load": False,
    "mixed_precision":  True , # TRUE BY DEFAULT , depending to GPU setting this to True might cause nan value problem in metrics
    "loss": PPYoloELoss(
        use_static_assigner=False,
        num_classes=len(dataset_params['classes']),
        reg_max=16
    ),
    "valid_metrics_list": [
        DetectionMetrics_050(
            score_thres=0.1,
            top_k_predictions=300,
            num_cls=len(dataset_params['classes']),
            normalize_targets=True,
            include_classwise_ap=True,  # also report AP@0.50 for each class, not just the overall mAP
            class_names=dataset_params['classes'],
            post_prediction_callback=PPYoloEPostPredictionCallback(
                score_threshold=0.01,
                nms_top_k=1000,
                max_predictions=300,
                nms_threshold=0.5
            )
        )
    ],
    "metric_to_watch": 'mAP@0.50'
}


In [ ]:
from super_gradients.training.utils.callbacks import Callback

class NotebookMetricsRecorder(Callback):
    def __init__(self):
        self.epoch_history = []

    @staticmethod
    def _to_float(value):
        if hasattr(value, "item"):
            value = value.item()
        return float(value)

    def _get_epoch_entry(self, epoch):
        for entry in self.epoch_history:
            if entry["epoch"] == epoch:
                return entry
        new_entry = {"epoch": epoch}
        self.epoch_history.append(new_entry)
        return new_entry

    def on_train_loader_end(self, context):
        if not context.metrics_dict:
            return
        epoch_entry = self._get_epoch_entry(context.epoch)
        for name, value in context.metrics_dict.items():
            epoch_entry[f"Train/{name}"] = self._to_float(value)

    def on_validation_loader_end(self, context):
        if not context.metrics_dict:
            return
        epoch_entry = self._get_epoch_entry(context.epoch)
        for name, value in context.metrics_dict.items():
            epoch_entry[f"Valid/{name}"] = self._to_float(value)

metrics_recorder = NotebookMetricsRecorder()
train_params.setdefault("phase_callbacks", []).append(metrics_recorder)


In [ ]:
trainer.train(
    model=model,
    training_params=train_params,
    train_loader=train_data,
    valid_loader=val_data
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

if not metrics_recorder.epoch_history:
    raise RuntimeError("Run the training cell first so that metrics_recorder captures epoch metrics.")

metrics_df = pd.DataFrame(metrics_recorder.epoch_history).sort_values("epoch").reset_index(drop=True)

# SuperGradients prefixes validation metrics with "Valid/"
wanted = [
    "Valid/Recall@0.50",
    "Valid/Precision@0.50",
    "Valid/mAP@0.50",
    "Valid/F1@0.50",
    "Valid/PPYoloELoss/loss",
]
value_columns = [c for c in wanted if c in metrics_df.columns]
# per-class AP (reported because include_classwise_ap=True in the validation metric)
value_columns += [c for c in metrics_df.columns if c.startswith("Valid/AP@0.50_")]

if not value_columns:
    raise RuntimeError(f"None of the requested metrics were found. Available: {list(metrics_df.columns)}")

metrics_to_plot = metrics_df[["epoch"] + value_columns]
display(metrics_to_plot)

for metric in value_columns:
    ax = metrics_to_plot.set_index("epoch")[[metric]].plot(figsize=(12, 6))
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Metric value")
    ax.set_title(metric)
    ax.grid(True, linestyle="--", alpha=0.5)
    plt.show()


In [ ]:
from super_gradients.training import models

from pathlib import Path
run_root = Path(CHECKPOINT_DIR) / EXPERIMENT_NAME
run_ids = sorted([p.name for p in run_root.glob("RUN_*")])
if not run_ids:
    raise FileNotFoundError(f"No RUN_* folders found in {run_root}")
RUN_ID = run_ids[-1]  # latest run
 
# ckpt_best.pth is the checkpoint with the best validation mAP@0.50, and the one
# batch_inference uses. (The run also writes average_model.pth, an average of the
# best snapshots, and ckpt_latest.pth, the last epoch, in the same folder.)
BEST_CHECKPOINT = run_root / RUN_ID / "ckpt_best.pth"
print(f"Loading {BEST_CHECKPOINT}")

best_model = models.get(
    MODEL_ARCH,
    num_classes=len(dataset_params['classes']),
    checkpoint_path=str(BEST_CHECKPOINT)
).to(DEVICE)


In [ ]:
import cv2
import random
from pathlib import Path
 
# pick a random validation image
val_images_dir = Path(dataset_params['data_dir']) / dataset_params['val_images_dir']
image_path = str(random.choice([p for p in val_images_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}]))
image = cv2.imread(image_path)
 
# predict
model_result = best_model.predict(image, conf=0.2)
 
print(model_result.prediction)


In [ ]:
import cv2
import matplotlib.pyplot as plt
 
label_dict = {i: name for i, name in enumerate(dataset_params["classes"])}
 
# Load the image (replace with your image loading method)
image = cv2.imread(image_path)
 
# Bounding boxes, labels, confidence, and label dictionary
bboxes = model_result.prediction.bboxes_xyxy
confidences = model_result.prediction.confidence
labels = model_result.prediction.labels
 
# Draw bounding boxes and labels on the image
for i in range(len(bboxes)):
    bbox = bboxes[i]
    confidence = confidences[i]
    label = labels[i]
     
    # Coordinates of the bounding box
    x1, y1, x2, y2 = [int(coord) for coord in bbox]
 
    # Draw the rectangle
    cv2.rectangle(image, (x1, y1), (x2, y2), color=(0, 255, 0), thickness=2)
 
    # Create label text with confidence
    label_text = f"{label_dict[label]}: {confidence:.2f}"
 
    # Put the label text above the bounding box
    cv2.putText(image, label_text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 
                fontScale=0.5, color=(255, 255, 255), thickness=1, lineType=cv2.LINE_AA)
 
# Convert BGR to RGB for displaying in matplotlib
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
 
# Display the image using matplotlib
plt.figure(figsize=(10, 10))
plt.imshow(image_rgb)
plt.axis('off')  # Turn off axis
plt.show()
